# RecSys 2026 — BM25 Baseline

A simple **BM25** sparse retrieval baseline for the Music-CRS challenge.

**Approach:**
1. Build a BM25 index over track metadata (name + artist + tags)
2. For each conversation turn, use the user utterance as query
3. Retrieve top-k tracks and evaluate with nDCG@{1, 10, 20}

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

DATA_DIR = Path('../data/raw')
print('Setup complete \u2705')

Setup complete ✅


## 1. Load Data

In [2]:
# Conversations
train_df = pd.read_parquet(DATA_DIR / 'TalkPlayData-Challenge-Dataset/data/train-00000-of-00001.parquet')
dev_df = pd.read_parquet(DATA_DIR / 'TalkPlayData-Challenge-Dataset/data/test-00000-of-00001.parquet')
print(f'Train sessions: {len(train_df):,} | Dev sessions: {len(dev_df):,}')

# Track catalog
tracks_df = pd.read_parquet(DATA_DIR / 'TalkPlayData-Challenge-Track-Metadata/data/all_tracks-00000-of-00001.parquet')
print(f'Track catalog: {len(tracks_df):,} tracks')

Train sessions: 15,199 | Dev sessions: 1,000
Track catalog: 47,071 tracks


## 2. Prepare Track Text for BM25

In [3]:
# Flatten list columns into strings for BM25 indexing
def flatten_list_col(val):
    """Convert list/ndarray to space-separated string."""
    if isinstance(val, (list, np.ndarray)):
        return ' '.join(str(v) for v in val)
    return str(val) if pd.notna(val) else ''

tracks_df['text_track_name'] = tracks_df['track_name'].apply(flatten_list_col)
tracks_df['text_artist_name'] = tracks_df['artist_name'].apply(flatten_list_col)
tracks_df['text_tags'] = tracks_df['tag_list'].apply(flatten_list_col)

# Combined document text
tracks_df['bm25_doc'] = (
    tracks_df['text_track_name'] + ' ' +
    tracks_df['text_artist_name'] + ' ' +
    tracks_df['text_tags']
).str.lower()

print(f'Sample BM25 doc: {tracks_df["bm25_doc"].iloc[0][:200]}...')

Sample BM25 doc: with rainy eyes emancipator relaxing experimental goeiepoep instrumental piano lo-fi body parts smooth life is easy top quality ambient downtempo instrumental conscious via pandora jazz chillout bluei...


## 3. Build BM25 Index

In [4]:
from rank_bm25 import BM25Okapi

# Tokenize corpus
corpus = [doc.split() for doc in tqdm(tracks_df['bm25_doc'], desc='Tokenizing')]
track_ids = tracks_df['track_id'].tolist()

# Build index
bm25 = BM25Okapi(corpus)
print(f'BM25 index built over {len(corpus):,} tracks \u2705')

Tokenizing:   0%|          | 0/47071 [00:00<?, ?it/s]

BM25 index built over 47,071 tracks ✅


## 4. Extract Ground Truth & Queries from Dev Set

In [5]:
def extract_turns(df):
    """Extract (session_id, turn_number, user_query, ground_truth_track_id) from conversations."""
    records = []
    for _, row in df.iterrows():
        session_id = row['session_id']
        convs = row['conversations']
        
        # Group by turn_number
        turns_by_num = {}
        for c in convs:
            tn = c['turn_number']
            if tn not in turns_by_num:
                turns_by_num[tn] = {}
            turns_by_num[tn][c['role']] = c['content']
        
        for tn, roles in sorted(turns_by_num.items()):
            if 'user' in roles and 'music' in roles:
                records.append({
                    'session_id': session_id,
                    'turn_number': tn,
                    'query': roles['user'],
                    'gt_track_id': roles['music'],
                })
    return pd.DataFrame(records)

dev_turns = extract_turns(dev_df)
print(f'Dev turns to evaluate: {len(dev_turns):,}')
dev_turns.head()

Dev turns to evaluate: 8,000


,session_id,turn_number,query,gt_track_id
0,ba3da7b0-1e81-4d2a-90fa-65ee1f4d7348,1,Play the highly popular grunge track from the ...,2445ed62-2e19-4222-8d01-3a57f685755d
1,ba3da7b0-1e81-4d2a-90fa-65ee1f4d7348,2,Perfect! That was the popular song I was looki...,9fe2712d-337b-4b9a-9a75-285b89e1543b
2,ba3da7b0-1e81-4d2a-90fa-65ee1f4d7348,3,"Yes, that's a great one! Arctic Monkeys always...",3d02bc2d-7c9a-4ea2-9361-b05b2fe654c1
3,ba3da7b0-1e81-4d2a-90fa-65ee1f4d7348,4,Another solid track from Arctic Monkeys. I app...,dfcc86b1-610b-4776-8084-dc2b3726e980
4,ba3da7b0-1e81-4d2a-90fa-65ee1f4d7348,5,"Yes, ""Mysterious Ways"" is a fantastic choice! ...",bb241864-6485-448f-9078-44ba8fb65438


## 5. Run BM25 Retrieval

In [ ]:
TOP_K = 20

def bm25_retrieve(query: str, top_k: int = TOP_K) -> list:
    """Retrieve top-k track_ids for a query."""
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    top_idx = scores.argsort()[::-1][:top_k]
    return [track_ids[i] for i in top_idx]

# Run retrieval on all dev turns
predictions = {}
ground_truths = {}

for _, row in tqdm(dev_turns.iterrows(), total=len(dev_turns), desc='BM25 retrieval'):
    sid = row['session_id']
    if sid not in predictions:
        predictions[sid] = []
        ground_truths[sid] = []
    
    recs = bm25_retrieve(row['query'])
    predictions[sid].append(recs)
    ground_truths[sid].append([row['gt_track_id']])

print(f'Predictions for {len(predictions)} sessions')

BM25 retrieval:   0%|          | 0/8000 [00:00<?, ?it/s]

## 6. Evaluate

In [ ]:
from src.evaluation.metrics import compute_ndcg, compute_catalog_coverage

# nDCG
ndcg_results = compute_ndcg(predictions, ground_truths, k_values=[1, 10, 20])
print('=== BM25 Baseline Results ===')
for metric, score in ndcg_results.items():
    print(f'  {metric}: {score:.4f}')

# Catalog coverage
coverage = compute_catalog_coverage(predictions, catalog_size=len(tracks_df))
print(f'  Catalog Coverage: {coverage:.4f}')

## 7. Quick Error Analysis

In [ ]:
# Check a few examples where BM25 succeeds or fails
track_lookup = tracks_df.set_index('track_id')

n_shown = 0
for _, row in dev_turns.head(20).iterrows():
    gt = row['gt_track_id']
    recs = bm25_retrieve(row['query'], top_k=5)
    hit = gt in recs
    
    if n_shown < 5:
        gt_name = track_lookup.loc[gt, 'track_name'] if gt in track_lookup.index else '??'
        print(f'Query: "{row["query"][:80]}..."')
        print(f'  GT: {gt_name} ({"HIT" if hit else "MISS"})')
        print(f'  Top-3 recs: {[track_lookup.loc[r, "track_name"] if r in track_lookup.index else r for r in recs[:3]]}')
        print()
        n_shown += 1